In [198]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
alert = Alert('1h','GGAL')

In [200]:
pre_df = yf.download(tickers='GGAL', period='200d', interval='1h')
pre_df = pre_df.iloc[:-3 , :]
display(pre_df)

C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1794724637.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  pre_df = yf.download(tickers='GGAL', period='200d', interval='1h')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GGAL,GGAL,GGAL,GGAL,GGAL
Datetime,,,,,
2024-09-04 13:30:00+00:00,41.230000,42.209999,40.279999,40.400002,659018
2024-09-04 14:30:00+00:00,42.025002,42.419899,41.189999,41.200001,451777
2024-09-04 15:30:00+00:00,42.290001,42.360001,41.740002,42.069500,263291
2024-09-04 16:30:00+00:00,42.009998,42.500000,41.950001,42.290001,343377
2024-09-04 17:30:00+00:00,41.224998,42.169998,41.070000,42.009998,328622
...,...,...,...,...,...
2025-06-20 18:30:00+00:00,52.230000,52.567501,52.230000,52.450001,92073
2025-06-20 19:30:00+00:00,51.939999,52.240002,51.930000,52.200001,391632


pre_df = alert.set_test(pre_df, "1h")

In [202]:
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.Close - df.Low.rolling(K).min()) /
        (df.High.rolling(K).max() - df.Low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df

In [203]:
def set_test( df, period):
        k = 17   
        d = 5    
        smth = 8   
        dayM = 5    
        semM = 4

        # Convert time
        df["time"] = pd.to_datetime(df.index, utc=True)
        df['timeArg'] = df['time'].dt.tz_convert('America/Argentina/Buenos_Aires')
        df['time'] = df['time'].dt.tz_convert(None)

        df = stochastic(df, "hora", k, d, smth)
        df = stochastic(df, "dia", k*dayM, d*dayM, smth*dayM)
        df = stochastic(df, "sem", k*dayM*semM, d*dayM*semM, smth*dayM*semM)
        #df = set_signals(df)
    
        return df
df = set_test(pre_df, "1h")
display(df )

C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1787476449.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)
C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1787476449.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)


Price,Close,High,Low,Open,Volume,time,timeArg,khora,dhora,kdia,ddia,ksem,dsem
Ticker,GGAL,GGAL,GGAL,GGAL,GGAL,,,,,,,,
Datetime,,,,,,,,,,,,,
2024-09-04 13:30:00+00:00,41.230000,42.209999,40.279999,40.400002,659018,2024-09-04 13:30:00,2024-09-04 10:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-04 14:30:00+00:00,42.025002,42.419899,41.189999,41.200001,451777,2024-09-04 14:30:00,2024-09-04 11:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-04 15:30:00+00:00,42.290001,42.360001,41.740002,42.069500,263291,2024-09-04 15:30:00,2024-09-04 12:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-04 16:30:00+00:00,42.009998,42.500000,41.950001,42.290001,343377,2024-09-04 16:30:00,2024-09-04 13:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-04 17:30:00+00:00,41.224998,42.169998,41.070000,42.009998,328622,2024-09-04 17:30:00,2024-09-04 14:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-20 18:30:00+00:00,52.230000,52.567501,52.230000,52.450001,92073,2025-06-20 18:30:00,2025-06-20 15:30:00-03:00,18.408288,34.598058,24.530830,26.295056,62.960834,71.146180
2025-06-20 19:30:00+00:00,51.939999,52.240002,51.930000,52.200001,391632,2025-06-20 19:30:00,2025-06-20 16:30:00-03:00,12.803984,26.561627,23.853753,26.282811,62.549378,71.005632


In [217]:
# Seleciona as linhas que possuem pelo menos um valor NaN
rows_with_nan = df[df.isna().any(axis=1)]
#display(rows_with_nan)

# Remove as linhas que possuem valores NaN e salva o resultado em um novo DataFrame (ou atualize o mesmo)
df_clean = df.dropna()
df_clean.columns = df_clean.columns.get_level_values(0)


display (df_clean)

# Se preferir modificar o DataFrame original, use inplace=True:
# df.dropna(inplace=True)

Price,Close,High,Low,Open,Volume,time,timeArg,khora,dhora,kdia,ddia,ksem,dsem
Datetime,,,,,,,,,,,,,
2025-01-07 17:30:00+00:00,73.900002,74.000000,73.050003,73.184998,113333,2025-01-07 17:30:00,2025-01-07 14:30:00-03:00,80.008716,78.640851,66.466258,57.018303,83.018518,84.866208
2025-01-07 18:30:00+00:00,73.110001,73.900002,72.970001,73.900002,128675,2025-01-07 18:30:00,2025-01-07 15:30:00-03:00,80.324670,78.793441,67.418153,57.770896,83.031453,84.826050
2025-01-07 19:30:00+00:00,71.860001,73.160004,71.449898,73.160004,121027,2025-01-07 19:30:00,2025-01-07 16:30:00-03:00,79.042488,79.157652,68.339545,58.555287,83.009144,84.784611
2025-01-07 20:30:00+00:00,72.279999,72.300003,71.532898,71.800003,154209,2025-01-07 20:30:00,2025-01-07 17:30:00-03:00,77.631602,79.066676,69.326225,59.359270,83.006705,84.742446
2025-01-08 14:30:00+00:00,70.559998,71.910004,69.690002,71.910004,212563,2025-01-08 14:30:00,2025-01-08 11:30:00-03:00,73.867888,78.175073,70.121200,60.163796,82.957320,84.699220
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-20 18:30:00+00:00,52.230000,52.567501,52.230000,52.450001,92073,2025-06-20 18:30:00,2025-06-20 15:30:00-03:00,18.408288,34.598058,24.530830,26.295056,62.960834,71.146180
2025-06-20 19:30:00+00:00,51.939999,52.240002,51.930000,52.200001,391632,2025-06-20 19:30:00,2025-06-20 16:30:00-03:00,12.803984,26.561627,23.853753,26.282811,62.549378,71.005632
2025-06-23 13:30:00+00:00,51.830002,52.320000,50.660000,51.685001,351493,2025-06-23 13:30:00,2025-06-23 10:30:00-03:00,9.554716,19.847252,23.497717,26.245171,62.090121,70.861727


In [219]:

def set_compra( df, i):
        df.loc[i, "oper"] = "COMPRA"


def set_venta( df, i):
        df.loc[i, "oper"] = "VENTA"


def close_long( df, i, compra, venta):
        df.loc[i, "oper"] = "CLOSELONG"
        if compra !=0.:
            df.loc[i, "long"] = (venta - compra) / compra


def close_short( df, i, compra, venta):
        df.loc[i, "oper"] = "CLOSESHORT"
        if venta !=0.:
            df.loc[i, "short"] = (venta - compra) / venta

In [221]:
i='2025-06-20 16:30:00+00:00'
l = df.loc[i]
#df.loc[i, "state"] = state
print(l)
print(l.khora)

Price    Ticker
Close    GGAL                      52.259998
High     GGAL                          52.43
Low      GGAL                      52.150002
Open     GGAL                      52.150002
Volume   GGAL                          69072
time                     2025-06-20 16:30:00
timeArg            2025-06-20 13:30:00-03:00
khora                              33.544893
dhora                              51.260381
kdia                                25.85307
ddia                               26.193056
ksem                               63.641393
dsem                               71.419199
Name: 2025-06-20 16:30:00+00:00, dtype: object
Ticker
    33.544893
Name: 2025-06-20 16:30:00+00:00, dtype: object


In [227]:
def set_signals(df):
        state = ""
        df["compra"] = False
        df["venta"] = False
        df["closeLong"] = False
        df["closeShort"] = False
        df["entradaLongH"] = False
        df["entradaShortH"] = False
        df["oper"] = ""
        df["long"] = ""
        df["short"] = ""
        df["state"] = "neutro"

        compra = 0.
        venta = 0.
#alert.py part 2
        
# outros intervalos
        lastEntradaLongH = False
        lastEntradaShortH = False
            
        for i in df.index:
            l = df.loc[i]
            df.loc[i, "state"] = state

        #for i, l in df.iterrows():
            #df.at[i, "state"] = state
            khora = float(l.khora)
            dhora = float(l.dhora)
            kdia = float(l.kdia)
            ddia = float(l.ddia)
            ksem = float(l.ksem)
            dsem = float(l.dsem)
            
            if (khora > 20) and (khora > dhora):
                    df.at[i, "entradaLongH"] = True
        
                # ESTRATEGIA 1 - Stochastico alineado y esperar cruce
                #if (l.khora > 20. and l.khora > l.dhora):
                    #df.loc[i, "entradaLongH"] = True
                    if (not lastEntradaLongH):
                        if (kdia > 20. and kdia > ddia):
                            if(state == "venta"):
                                df.loc[i, "closeShort"] = True
                                state = "neutro"
                                compra = l.Close
                                close_short(df, i, compra, venta)
                            if (ksem > 80. or ksem > dsem):
                                df.loc[i, "compra"] = True
                                if state != "compra":
                                    state = "compra"
                                    compra = l.Close
                                    set_compra(df, i)
            else:
                df.loc[i, "entradaLongH"] = False

            if (l.khora < 80. and l.khora < l.dhora):
                    df.loc[i, "entradaShortH"] = True
                    if (not lastEntradaShortH):
                        if(l.kdia < 80. and l.kdia < l.ddia):
                            if(state == "compra"):
                                df.loc[i, "closeLong"] = True
                                state = "neutro"
                                venta = l.Close
                                close_long(df, i, compra, venta)
                            if(l.ksem < 20. or l.ksem < l.dsem):
                                df.loc[i, "venta"] = True
                                if state != "venta":
                                    state = "venta"
                                    venta = l.Close
                                    set_venta(df, i)
            else:
                    df.loc[i, "entradaShortH"] = False

            lastEntradaLongH = df.loc[i, "entradaLongH"]
            lastEntradaShortH = df.loc[i, "entradaShortH"]
                    
        
        return df
df= set_signals(df_clean)          
display(df)

Price,Close,High,Low,Open,Volume,time,timeArg,khora,dhora,kdia,...,compra,venta,closeLong,closeShort,entradaLongH,entradaShortH,oper,long,short,state
Datetime,,,,,,,,,,,,,,,,,,,,,
2025-01-07 17:30:00+00:00,73.900002,74.000000,73.050003,73.184998,113333,2025-01-07 17:30:00,2025-01-07 14:30:00-03:00,80.008716,78.640851,66.466258,...,True,False,False,False,True,False,COMPRA,,,
2025-01-07 18:30:00+00:00,73.110001,73.900002,72.970001,73.900002,128675,2025-01-07 18:30:00,2025-01-07 15:30:00-03:00,80.324670,78.793441,67.418153,...,False,False,False,False,True,False,,,,compra
2025-01-07 19:30:00+00:00,71.860001,73.160004,71.449898,73.160004,121027,2025-01-07 19:30:00,2025-01-07 16:30:00-03:00,79.042488,79.157652,68.339545,...,False,False,False,False,False,True,,,,compra
2025-01-07 20:30:00+00:00,72.279999,72.300003,71.532898,71.800003,154209,2025-01-07 20:30:00,2025-01-07 17:30:00-03:00,77.631602,79.066676,69.326225,...,False,False,False,False,False,True,,,,compra
2025-01-08 14:30:00+00:00,70.559998,71.910004,69.690002,71.910004,212563,2025-01-08 14:30:00,2025-01-08 11:30:00-03:00,73.867888,78.175073,70.121200,...,False,False,False,False,False,True,,,,compra
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-20 18:30:00+00:00,52.230000,52.567501,52.230000,52.450001,92073,2025-06-20 18:30:00,2025-06-20 15:30:00-03:00,18.408288,34.598058,24.530830,...,False,False,False,False,False,True,,,,neutro
2025-06-20 19:30:00+00:00,51.939999,52.240002,51.930000,52.200001,391632,2025-06-20 19:30:00,2025-06-20 16:30:00-03:00,12.803984,26.561627,23.853753,...,False,False,False,False,False,True,,,,neutro
2025-06-23 13:30:00+00:00,51.830002,52.320000,50.660000,51.685001,351493,2025-06-23 13:30:00,2025-06-23 10:30:00-03:00,9.554716,19.847252,23.497717,...,False,False,False,False,False,True,,,,neutro


In [234]:
last = df.iloc [-1]
display (last)

Price
Close                            51.139999
High                             51.564999
Low                              51.099998
Open                             51.529999
Volume                              120909
time                   2025-06-23 15:30:00
timeArg          2025-06-23 12:30:00-03:00
khora                            10.991914
dhora                            12.481418
kdia                             22.497046
ddia                             26.073161
ksem                             61.130318
dsem                             70.562807
compra                               False
venta                                False
closeLong                            False
closeShort                           False
entradaLongH                         False
entradaShortH                         True
oper                                      
long                                      
short                                     
state                               neutro
Name:

In [239]:
lastOperCloseLong = df[df['oper'] == 'CLOSELONG'].iloc[-1]
display(lastOperCloseLong)
lastOperCloseShort = df[df['oper'] == 'CLOSESHORT'].iloc[-1]
display(lastOperCloseShort)

Price
Close                            61.325001
High                                  62.0
Low                              60.970001
Open                                 61.98
Volume                              245143
time                   2025-05-27 17:30:00
timeArg          2025-05-27 14:30:00-03:00
khora                            59.275209
dhora                            62.504512
kdia                             72.956238
ddia                             76.546611
ksem                             79.143294
dsem                             73.497593
compra                               False
venta                                False
closeLong                             True
closeShort                           False
entradaLongH                         False
entradaShortH                         True
oper                             CLOSELONG
long                             -0.004626
short                                     
state                               compra
Name:

Price
Close                            54.654999
High                                 54.73
Low                              53.615002
Open                             53.615002
Volume                              183788
time                   2025-06-17 15:30:00
timeArg          2025-06-17 12:30:00-03:00
khora                            27.628312
dhora                            24.838737
kdia                             26.079092
ddia                             22.078226
ksem                              67.14329
dsem                             73.281018
compra                               False
venta                                False
closeLong                            False
closeShort                            True
entradaLongH                          True
entradaShortH                        False
oper                            CLOSESHORT
long                                      
short                             0.036322
state                                venta
Name:

In [248]:

if lastOperCloseLong.time > lastOperCloseShort.time:
    missingAlert = lastOperCloseLong
else:
    missingAlert = lastOperCloseShort

display(missingAlert)

Price
Close                            54.654999
High                                 54.73
Low                              53.615002
Open                             53.615002
Volume                              183788
time                   2025-06-17 15:30:00
timeArg          2025-06-17 12:30:00-03:00
khora                            27.628312
dhora                            24.838737
kdia                             26.079092
ddia                             22.078226
ksem                              67.14329
dsem                             73.281018
compra                               False
venta                                False
closeLong                            False
closeShort                            True
entradaLongH                          True
entradaShortH                        False
oper                            CLOSESHORT
long                                      
short                             0.036322
state                                venta
Name:

In [51]:
df = stochastic(pre_df,"1h",17,5,8)
last = df.iloc[-1]
display (last)

# Read last alert csv
lastAlert = alert.read_alert_csv('lastAlert_1h.csv')
display(lastAlert)

C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1567423116.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)


Price   Ticker
Close   GGAL         55.020000
High    GGAL         55.299999
Low     GGAL         54.919998
Open    GGAL         55.090000
Volume  GGAL      80081.000000
k1h                  63.429191
d1h                  51.514441
Name: 2025-06-18 16:30:00+00:00, dtype: float64

Unnamed: 0
Open                      8.69849967956543
High                     8.930000305175781
Low                      8.680000305175781
Close                    8.869999885559082
Adj Close                8.869999885559082
Volume                              147519
time                   2022-12-21 16:30:00
timeArg          2022-12-21 13:30:00-03:00
khora                    89.27899126024955
dhora                     89.2503455622804
kdia                    51.219426452147296
ddia                     41.25813962855897
ksem                     44.75414109800853
dsem                     32.42956752293771
compra                                True
venta                                False
closeLong                            False
closeShort                           False
entradaLongH                          True
entradaShortH                        False
oper                                COMPRA
long                                   NaN
short                                  NaN
